In [1]:
import os
import pickle
import networkx as nx
import sys
import pandas as pd

# Determine the project root directory for relative imports
try:
    # This will work in scripts where __file__ is defined
    current_dir = os.path.dirname(os.path.abspath(__file__))
    # Assuming "src" is parallel to the script folder
    project_root = os.path.abspath(os.path.join(current_dir, ".."))
except NameError:
    # In notebooks __file__ is not defined: assume we're in notebooks/riziv_dataset/
    project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Local application imports


In [2]:
# Define the path to the BSARD dataset files
BSARD_data_path = os.path.join(project_root, "data", "BSARD_dataset")

bsard_corpus = pd.read_csv(os.path.join(BSARD_data_path, 'inputs', 'bsard_corpus.csv'))

bsard_corpus.head(3)

,id,reference,article,law_type,code,book,part,act,chapter,section,subsection,description
0,1,"Art. 1.1.1, Code Bruxellois de l'Air, du Clima...",Le présent Code règle une matière visée à l'ar...,regional,"Code Bruxellois de l'Air, du Climat et de la M...",Dispositions communes,NaN,Généralités,NaN,NaN,NaN,"Dispositions communes, Généralités"
1,2,"Art. 1.1.2, Code Bruxellois de l'Air, du Clima...",Le présent Code transpose en Région de Bruxell...,regional,"Code Bruxellois de l'Air, du Climat et de la M...",Dispositions communes,NaN,Généralités,NaN,NaN,NaN,"Dispositions communes, Généralités"
2,3,"Art. 1.2.1, Code Bruxellois de l'Air, du Clima...",Le présent Code poursuit les objectifs suivant...,regional,"Code Bruxellois de l'Air, du Climat et de la M...",Dispositions communes,NaN,Objectifs,NaN,NaN,NaN,"Dispositions communes, Objectifs"


In [3]:
base_document_graph_path = os.path.join(BSARD_data_path, 'intermediate', 'base_document_graph_B.pkl')

# Carga del grafo
with open(base_document_graph_path, 'rb') as f:
    G = pickle.load(f)

In [4]:
non_document_entities_path = os.path.join(BSARD_data_path, 'intermediate', 'keyword_extraction.pkl')

# Carga del diccionario de keywords
with open(non_document_entities_path, 'rb') as f:
    keywords_dict = pickle.load(f)

In [5]:
###############
# Parse retrieved keyword-related content and build dataframe
###############

# Keep only those articles which have been already scanned
keywords_dict_filter = {k:v for k,v in keywords_dict.items() if v != None} 

parsed_key_terms = []

for art in keywords_dict_filter.keys():

    for n in ["1", "2", "3", "4"]:

        parsed_key_terms.append((art, keywords_dict_filter[art][f"key_concept_{n}"]))

# Build Dataframe
key_terms_df = pd.DataFrame(parsed_key_terms, columns=['article_code', 'key_term'])

In [6]:
filtered_counts = key_terms_df["key_term"].value_counts()[key_terms_df["key_term"].value_counts() > 4]

In [7]:
filtered_counts

key_term
AWIPH                        256
Gouvernement                 138
Code décrétal                131
assemblée générale           123
procès-verbal                120
                            ... 
solutions de substitution      5
droits civils                  5
acte de dépôt                  5
expulsion                      5
Retrait de l'agrément          5
Name: count, Length: 2721, dtype: int64

In [8]:
filtered_keyterms_df = key_terms_df[key_terms_df['key_term'].isin(filtered_counts.index)]
filtered_keyterms_df = filtered_keyterms_df.reset_index(drop=True)

In [9]:
filtered_keyterms_df

,article_code,key_term
0,1.1.1.1,Constitution
1,1.1.1.1,présent Code
2,1.1.3.1,Région
3,1.1.3.1,Gouvernement
4,1.1.4.4,évaluation environnementale
...,...,...
30301,35.0.7.9,majorité prévue à l'article 4
30302,35.0.7.10,Trésor public
30303,35.0.7.11,Cour des comptes
30304,35.0.8.1,droits et obligations


In [10]:
df_keyterms_condensed = (
    key_terms_df
    .groupby('key_term')
    .agg(
        article_codes=('article_code', list),
        count=('article_code', 'size'),
        law_count=('article_code',lambda s: s.str.split('.').str[0].nunique())
    )
    .reset_index()
)

df_keyterms_condensed = df_keyterms_condensed[df_keyterms_condensed['count'] > 4] # At least 5 mentions in distinc articles
df_keyterms_condensed.reset_index(drop=True, inplace=True)
df_keyterms_condensed_more_laws = df_keyterms_condensed[df_keyterms_condensed['law_count'] > 2] # At least 3 different laws
df_keyterms_condensed_more_laws

,key_term,article_codes,count,law_count
0,1er janvier,"[16.2.4.6, 16.7.3.2, 20.1.3.3, 27.0.6.73, 27.2...",5,3
2,ASBL,"[17.9.8.3, 17.7.6.32, 28.2.3.52, 28.2.3.53, 28...",11,3
6,Accusé de réception,"[2.0.4.36, 2.0.4.69, 2.0.8.9, 14.2.0.95, 16.6....",9,6
7,Acte authentique,"[4.2.12.5, 4.3.2.36, 31.2.2.20, 32.5.6.1, 32.9...",7,3
9,Action publique,"[4.2.8.32, 10.2.3.38, 24.1.0.18, 31.4.1.69, 34...",5,5
...,...,...,...,...
2711,évaluation des incidences,"[1.2.5.1, 2.0.2.51, 2.0.4.47, 2.0.4.66, 20.0.2...",17,5
2712,évaluation des incidences environnementales,"[20.0.0.27, 20.1.0.36, 23.5.2.48, 23.8.4.3, 23...",11,3
2715,évaluation environnementale,"[1.1.4.4, 2.0.3.2, 20.0.0.34, 23.3.1.3, 23.3.1...",10,4
2718,évaluation intermédiaire,"[10.2.9.130, 19.1.5.21, 19.1.8.12, 28.1.2.109,...",5,4


In [11]:
top10 = df_keyterms_condensed_more_laws.nlargest(10, 'law_count')[['key_term','law_count']]
top10

,key_term,law_count
2169,recours,21
2043,procès-verbal,20
2142,rapport annuel,19
2102,présent chapitre,18
88,Code judiciaire,17
447,amende administrative,17
1775,notification de la décision,17
547,autorisation préalable,16
1773,notification,16
1905,personne morale,16


In [12]:
from collections import Counter
import networkx as nx

# Basic counts
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

# Count nodes by type
node_types = [data.get('node_type', 'Unknown') for _, data in G.nodes(data=True)]
nodes_by_type = Counter(node_types)

# Degree metrics
degrees = dict(G.degree())
avg_degree = sum(degrees.values()) / num_nodes if num_nodes else 0
max_degree = max(degrees.values()) if degrees else 0
min_degree = min(degrees.values()) if degrees else 0

# (Optional) If directed, you can also look at in/out-degree:
if G.is_directed():
    in_deg = dict(G.in_degree())
    out_deg = dict(G.out_degree())
    avg_in_degree = sum(in_deg.values()) / num_nodes
    avg_out_degree = sum(out_deg.values()) / num_nodes

# Assemble and print summary
summary = {
    "total_nodes": num_nodes,
    "total_edges": num_edges,
    "nodes_by_type": dict(nodes_by_type),
    "avg_degree": avg_degree,
    "max_degree": max_degree,
    "min_degree": min_degree,
}

if G.is_directed():
    summary.update({
        "avg_in_degree": avg_in_degree,
        "avg_out_degree": avg_out_degree
    })

print("Graph summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

Graph summary:
  total_nodes: 22806
  total_edges: 90484
  nodes_by_type: {'Act': 35, 'Book': 150, 'Article': 22621}
  avg_degree: 7.93510479698325
  max_degree: 3100
  min_degree: 2
  avg_in_degree: 3.967552398491625
  avg_out_degree: 3.967552398491625


In [13]:
# Assume your DataFrame is named df_keyterms
# and the column containing lists of code strings is "article_codes".

def simplify_article_codes(code_list):
    simplified = []
    for code in code_list:
        parts = code.split('.')           # e.g. ["16", "2", "4", "6"]
        if len(parts) == 4:
            # drop the third element (index 2)
            new_code = ".".join([parts[0], parts[1], parts[3]])
            simplified.append(new_code)
        else:
            # leave codes with unexpected format unchanged
            simplified.append(code)
    return simplified

# Apply the transformation in-place:
df_keyterms_condensed_more_laws['article_codes'] = (
    df_keyterms_condensed_more_laws['article_codes']
    .apply(simplify_article_codes)
)


/tmp/ipykernel_87126/1825858460.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_keyterms_condensed_more_laws['article_codes'] = (


In [14]:
df_keyterms_condensed_more_laws

,key_term,article_codes,count,law_count
0,1er janvier,"[16.2.6, 16.7.2, 20.1.3, 27.0.73, 27.2.67]",5,3
2,ASBL,"[17.9.3, 17.7.32, 28.2.52, 28.2.53, 28.2.54, 2...",11,3
6,Accusé de réception,"[2.0.36, 2.0.69, 2.0.9, 14.2.95, 16.6.136, 20....",9,6
7,Acte authentique,"[4.2.5, 4.3.36, 31.2.20, 32.5.1, 32.9.4, 32.12...",7,3
9,Action publique,"[4.2.32, 10.2.38, 24.1.18, 31.4.69, 34.3.10]",5,5
...,...,...,...,...
2711,évaluation des incidences,"[1.2.1, 2.0.51, 2.0.47, 2.0.66, 20.0.1, 20.0.2...",17,5
2712,évaluation des incidences environnementales,"[20.0.27, 20.1.36, 23.5.48, 23.8.3, 23.8.4, 23...",11,3
2715,évaluation environnementale,"[1.1.4, 2.0.2, 20.0.34, 23.3.3, 23.3.4, 23.5.2...",10,4
2718,évaluation intermédiaire,"[10.2.130, 19.1.21, 19.1.12, 28.1.109, 29.1.11]",5,4


In [15]:
# Assuming you have:
# - summary_df: a DataFrame with columns 'key_term' and 'article_codes'
# - G: your pre-built hierarchical graph of Laws → Books → Chapters → Articles

# 1) Add each key term as a new node with node_type="KeyTerm"
G.add_nodes_from(
    (key_term, {"node_type": "KeyTerm"})
    for key_term in df_keyterms_condensed_more_laws['key_term']
)

# 2) Create edges between articles and key terms:
#    article -> key_term  with relation="cites"
#    key_term -> article  with relation="cited_in"
G.add_edges_from(
    (article_code, key_term, {"relation": "cites"})
    for key_term, codes in zip(df_keyterms_condensed_more_laws['key_term'], df_keyterms_condensed_more_laws['article_codes'])
    for article_code in codes
)
G.add_edges_from(
    (key_term, article_code, {"relation": "cited_in"})
    for key_term, codes in zip(df_keyterms_condensed_more_laws['key_term'], df_keyterms_condensed_more_laws['article_codes'])
    for article_code in codes
)


In [16]:
from collections import Counter
import networkx as nx

# Basic counts
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

# Count nodes by type
node_types = [data.get('node_type', 'Unknown') for _, data in G.nodes(data=True)]
nodes_by_type = Counter(node_types)

# Degree metrics
degrees = dict(G.degree())
avg_degree = sum(degrees.values()) / num_nodes if num_nodes else 0
max_degree = max(degrees.values()) if degrees else 0
min_degree = min(degrees.values()) if degrees else 0

# (Optional) If directed, you can also look at in/out-degree:
if G.is_directed():
    in_deg = dict(G.in_degree())
    out_deg = dict(G.out_degree())
    avg_in_degree = sum(in_deg.values()) / num_nodes
    avg_out_degree = sum(out_deg.values()) / num_nodes

# Assemble and print summary
summary = {
    "total_nodes": num_nodes,
    "total_edges": num_edges,
    "nodes_by_type": dict(nodes_by_type),
    "avg_degree": avg_degree,
    "max_degree": max_degree,
    "min_degree": min_degree,
}

if G.is_directed():
    summary.update({
        "avg_in_degree": avg_in_degree,
        "avg_out_degree": avg_out_degree
    })

print("Graph summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

Graph summary:
  total_nodes: 24089
  total_edges: 125440
  nodes_by_type: {'Act': 35, 'Book': 150, 'Article': 22621, 'KeyTerm': 1283}
  avg_degree: 10.414712109261488
  max_degree: 3100
  min_degree: 2
  avg_in_degree: 5.207356054630744
  avg_out_degree: 5.207356054630744


In [17]:
G.nodes["1er janvier"]

{'node_type': 'KeyTerm'}

In [18]:
list(G.neighbors("1er janvier"))

['16.2.6', '16.7.2', '20.1.3', '27.0.73', '27.2.67']

In [19]:
list(G.neighbors("1er janvier"))

['16.2.6', '16.7.2', '20.1.3', '27.0.73', '27.2.67']

In [21]:
list(G.neighbors('20.1.3'))

['20.1',
 '20.1.2',
 '20.1.4',
 '1er janvier',
 'autorité compétente',
 'autorités communales',
 'enquête publique',
 'notification',
 'reconnaissance',
 "étude d'incidences"]

In [22]:
len(G.nodes())

24089

In [23]:
from collections import Counter
import networkx as nx

# Basic counts
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

# Count nodes by type
node_types = [data.get('node_type', 'Unknown') for _, data in G.nodes(data=True)]
nodes_by_type = Counter(node_types)

# Degree metrics
degrees = dict(G.degree())
avg_degree = sum(degrees.values()) / num_nodes if num_nodes else 0
max_degree = max(degrees.values()) if degrees else 0
min_degree = min(degrees.values()) if degrees else 0

# (Optional) If directed, you can also look at in/out-degree:
if G.is_directed():
    in_deg = dict(G.in_degree())
    out_deg = dict(G.out_degree())
    avg_in_degree = sum(in_deg.values()) / num_nodes
    avg_out_degree = sum(out_deg.values()) / num_nodes

# Assemble and print summary
summary = {
    "total_nodes": num_nodes,
    "total_edges": num_edges,
    "nodes_by_type": dict(nodes_by_type),
    "avg_degree": avg_degree,
    "max_degree": max_degree,
    "min_degree": min_degree,
}

if G.is_directed():
    summary.update({
        "avg_in_degree": avg_in_degree,
        "avg_out_degree": avg_out_degree
    })

print("Graph summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")


Graph summary:
  total_nodes: 24089
  total_edges: 125440
  nodes_by_type: {'Act': 35, 'Book': 150, 'Article': 22621, 'KeyTerm': 1283}
  avg_degree: 10.414712109261488
  max_degree: 3100
  min_degree: 2
  avg_in_degree: 5.207356054630744
  avg_out_degree: 5.207356054630744


## 5. Save output

In [26]:
with open(os.path.join(BSARD_data_path, 'intermediate', "hybrid_graph_full_B.pkl"), 'wb') as f:
    pickle.dump(G, f)